# 04_207 · Comparación final de modelos con cuatro categorías

Este cuaderno no entrena. Reúne únicamente evaluaciones realizadas sobre validation/test 4:1 comunes y verifica que todos los resultados declaren el mismo hash de dataset. Debe ejecutarse después de los experimentos que se quieran comparar.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts_auxiliares import entrenar_qwen_acoso_amenaza as q4
from scripts_auxiliares import entrenar_transformers_planos_4 as t4
from scripts_auxiliares import experimentos_jerarquicos_4 as h4
from scripts_auxiliares import experimentos_jerarquicos_clasicos_4 as c4
from scripts_auxiliares import experimentos_qwen_jerarquico_4 as qh
EXPECTED_DATASET_SHA256 = 'df2ac01183271e44b6dcfb9cb4850bd6b1ef1cd11d9fc51c881be944670ef20f'
OUTPUT_DIR = ROOT / 'resultados' / 'metricas' / 'comparacion_final_4'
FIGURE_DIR = ROOT / 'resultados' / 'figuras' / 'comparacion_final_4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Descubrimiento y control del dataset

In [ ]:
frames = []
hashes = []
missing = []

def add_table(path, family, structure, dataset_hash):
    path = Path(path)
    if not path.exists():
        missing.append(str(path.relative_to(ROOT)))
        return
    table = pd.read_csv(path)
    table['familia'] = family
    table['estructura'] = structure
    table['fuente'] = str(path.relative_to(ROOT))
    frames.append(table)
    hashes.append((str(path.relative_to(ROOT)), dataset_hash))

if q4.EVALUATION_PATH.exists():
    evaluation = json.loads(q4.EVALUATION_PATH.read_text(encoding='utf-8'))
    rows = []
    for split, values in evaluation['metrics'].items():
        rows.append({'model_key':'qwen4_flat','modelo':'Qwen3-0.6B LoRA plano','split':split, **{k:v for k,v in values.items() if k != 'category_recall'}})
    table = pd.DataFrame(rows)
    table['familia']='Qwen'; table['estructura']='plano'; table['fuente']=str(q4.EVALUATION_PATH.relative_to(ROOT))
    frames.append(table); hashes.append((str(q4.EVALUATION_PATH.relative_to(ROOT)), evaluation['dataset_sha256']))
else:
    missing.append(str(q4.EVALUATION_PATH.relative_to(ROOT)))

if t4.RESULT_PATH.exists():
    result=json.loads(t4.RESULT_PATH.read_text(encoding='utf-8'))
    add_table(t4.METRICS_DIR/'comparacion.csv','Transformer','plano',result['dataset']['sha256'])
else: missing.append(str(t4.RESULT_PATH.relative_to(ROOT)))

if c4.RESULT_PATH.exists():
    result=json.loads(c4.RESULT_PATH.read_text(encoding='utf-8'))
    add_table(c4.COMMON_4A1_PATH,'Clásico','plano/cascada/jerárquico',result['dataset']['balanced_dataset_sha256'])
else: missing.append(str(c4.RESULT_PATH.relative_to(ROOT)))

for key, structure in [(h4.CASCADE_EXTRA_SAFE_KEY,'cascada'),(h4.JOINT_KEY,'multitarea')]:
    result_path=h4.result_path(key)
    if result_path.exists():
        result=json.loads(result_path.read_text(encoding='utf-8'))
        add_table(h4._experiment_paths(key)['comparison'],'Transformer',structure,result['dataset']['sha256'])
    else: missing.append(str(result_path.relative_to(ROOT)))

if qh.RESULT_PATH.exists():
    result=json.loads(qh.RESULT_PATH.read_text(encoding='utf-8'))
    add_table(qh.METRICS_DIR/'comparacion.csv','Qwen','cascada/multitarea',result['dataset']['sha256'])
else: missing.append(str(qh.RESULT_PATH.relative_to(ROOT)))

bad_hashes=[item for item in hashes if item[1] != EXPECTED_DATASET_SHA256]
if bad_hashes: raise ValueError(f'Resultados con otro dataset: {bad_hashes}')
display(pd.DataFrame(hashes,columns=['fuente','dataset_sha256']))
display(Markdown('**Pendientes:** ' + (', '.join(missing) if missing else 'ninguno')))

## 2. Tabla comparable

Se eliminan referencias duplicadas que algunos cuadernos incluyen como control. La selección global usa PR-AUC macro de validation; test sólo describe el modelo seleccionado y las demás alternativas.

In [ ]:
if not frames: raise RuntimeError('Todavía no hay experimentos terminados.')
comparison=pd.concat(frames,ignore_index=True,sort=False)
required=['modelo','split','damage_pr_auc_macro','damage_f1_macro','any_damage_recall','missed_damage_as_safe']
comparison=comparison.dropna(subset=[c for c in required if c in comparison]).copy()
comparison=comparison.drop_duplicates(subset=['modelo','split'],keep='last')
comparison.to_csv(OUTPUT_DIR/'comparacion_todos_modelos_4.csv',index=False)
validation=comparison.loc[comparison['split'].eq('validation')].sort_values(['damage_pr_auc_macro','damage_f1_macro'],ascending=False)
winner=str(validation.iloc[0]['modelo'])
test=comparison.loc[comparison['split'].eq('test')].sort_values('damage_pr_auc_macro',ascending=False)
display(Markdown(f'### Ganador por validation: **{winner}**'))
display(validation[['modelo','familia','estructura','damage_pr_auc_macro','damage_f1_macro','any_damage_recall']])
display(test[['modelo','familia','estructura','damage_pr_auc_macro','damage_f1_macro','any_damage_recall','missed_damage_as_safe']])

In [ ]:
plot=test.set_index('modelo')[['damage_pr_auc_macro','damage_f1_macro','any_damage_recall']]
ax=plot.plot.bar(figsize=(14,6))
ax.set_ylim(0,1)
ax.set_title('Comparación sobre el mismo test 4:1')
ax.grid(axis='y',alpha=.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR/'comparacion_todos_modelos_4.png',dpi=180,bbox_inches='tight')
plt.show()

## Interpretación

El ranking global por validation no sustituye los contrastes pareados de cada cuaderno jerárquico. Para producción también deben revisarse recall mínimo por categoría, falsos negativos, tasa de revisión y los intervalos por video. Ningún modelo se considera autónomo sin gold standard humano independiente y piloto prospectivo.